# 07 - Inactivity Detection
Part A: Activity scoring from customer data. Part B: Bank transactions standalone analysis.

In [1]:
import pandas as pd, numpy as np, joblib
from sklearn.preprocessing import MinMaxScaler
import plotly.express as px
import warnings; warnings.filterwarnings('ignore')
SEED=42; np.random.seed(SEED)
PRIMARY='#635BFF'; RISK='#E74C3C'; SAFE='#27AE60'; NEUTRAL='#3498DB'; WARNING='#F39C12'; TEMPLATE='plotly_white'


## Part A: Inactivity Scoring from Customer Data

In [2]:
df = pd.read_csv('data/processed/customer_clean.csv')
print(f"Shape: {df.shape}")
cols_used = ['CLIENTNUM','Months_Inactive_12_mon','Total_Trans_Ct','Total_Trans_Amt','Total_Revolving_Bal','Avg_Utilization_Ratio']
print("Using columns:", cols_used)


Shape: (10127, 22)
Using columns: ['CLIENTNUM', 'Months_Inactive_12_mon', 'Total_Trans_Ct', 'Total_Trans_Amt', 'Total_Revolving_Bal', 'Avg_Utilization_Ratio']


In [3]:
def minmax(s):
    """Min-max normalize a pandas Series."""
    return (s - s.min()) / (s.max() - s.min())

df['norm_trans_ct']    = minmax(df['Total_Trans_Ct'])
df['norm_trans_amt']   = minmax(df['Total_Trans_Amt'])
df['norm_revolving']   = minmax(df['Total_Revolving_Bal'])
df['norm_utilization'] = minmax(df['Avg_Utilization_Ratio'])
df['norm_inactivity']  = minmax(df['Months_Inactive_12_mon'])

df['activity_score'] = (
    0.30 * (1 - df['norm_inactivity']) +
    0.30 * df['norm_trans_ct'] +
    0.20 * df['norm_trans_amt'] +
    0.10 * df['norm_revolving'] +
    0.10 * df['norm_utilization']
)

def assign_category(score):
    """Assign activity category based on score thresholds."""
    if score >= 0.70: return "Active"
    elif score >= 0.40: return "Moderately Active"
    elif score >= 0.20: return "Inactive"
    else: return "High Risk"

df['activity_category'] = df['activity_score'].apply(assign_category)
df['future_churn_candidate'] = (
    (df['Months_Inactive_12_mon'] >= 3) &
    (df['Avg_Utilization_Ratio'] < 0.15)
)

print("Activity category distribution:")
print(df['activity_category'].value_counts())
print(f"\nFuture churn candidates: {df['future_churn_candidate'].sum()}")


Activity category distribution:
activity_category
Moderately Active    5681
Inactive             4085
Active                191
High Risk             170
Name: count, dtype: int64

Future churn candidates: 2197


In [4]:
cat_counts = df['activity_category'].value_counts().reset_index()
cat_counts.columns = ['category','count']
fig = px.bar(cat_counts, x='category', y='count', color='category',
             color_discrete_map={'Active':SAFE,'Moderately Active':NEUTRAL,'Inactive':WARNING,'High Risk':RISK},
             template=TEMPLATE, title='Activity Category Distribution')
fig.show()


In [5]:
fig = px.histogram(df, x='activity_score', nbins=40, color_discrete_sequence=[PRIMARY],
                   template=TEMPLATE, title='Activity Score Distribution')
fig.show()


In [6]:
fc = df['future_churn_candidate'].value_counts().reset_index()
fc.columns = ['candidate','count']
fc['candidate'] = fc['candidate'].map({True:'Yes',False:'No'})
fig = px.bar(fc, x='candidate', y='count', color='candidate',
             color_discrete_map={'Yes':RISK,'No':SAFE},
             template=TEMPLATE, title='Future Churn Candidates')
fig.show()


In [7]:
fig = px.scatter(df.sample(2000, random_state=SEED), x='Months_Inactive_12_mon', y='activity_score',
                 color='activity_category',
                 color_discrete_map={'Active':SAFE,'Moderately Active':NEUTRAL,'Inactive':WARNING,'High Risk':RISK},
                 template=TEMPLATE, title='Inactivity Months vs Activity Score', opacity=0.6)
fig.show()


In [8]:
if 'Attrition_Flag' in df.columns:
    ct = df.groupby(['activity_category','Attrition_Flag']).size().reset_index(name='count')
    fig = px.bar(ct, x='activity_category', y='count', color='Attrition_Flag', barmode='group',
                 color_discrete_map={'Existing Customer':SAFE,'Attrited Customer':RISK},
                 template=TEMPLATE, title='Activity Category vs Churn Status')
    fig.show()


In [9]:
# Save Part A
inactivity_df = df[['CLIENTNUM','activity_score','activity_category','future_churn_candidate']]
inactivity_df.to_csv('data/processed/inactivity_scores.csv', index=False)
print(f"Saved inactivity_scores.csv - shape: {inactivity_df.shape}")

# Save scaler as model
import os; os.makedirs('models/inactivity', exist_ok=True)
mms = MinMaxScaler()
mms.fit(df[['Total_Trans_Ct','Total_Trans_Amt','Total_Revolving_Bal','Avg_Utilization_Ratio','Months_Inactive_12_mon']])
joblib.dump(mms, 'models/inactivity/activity_scorer.pkl')
print("Saved activity_scorer.pkl")


Saved inactivity_scores.csv - shape: (10127, 4)
Saved activity_scorer.pkl


## Part B: Bank Transactions Standalone Analysis

In [10]:
btx = pd.read_csv('data/raw/bank_transactions/bank_transactions.csv')
print(f"Shape: {btx.shape}")
btx['TransactionDate'] = pd.to_datetime(btx['TransactionDate'])
btx['PreviousTransactionDate'] = pd.to_datetime(btx['PreviousTransactionDate'])
print(btx.dtypes)


Shape: (2512, 16)
TransactionID                         str
AccountID                             str
TransactionAmount                 float64
TransactionDate            datetime64[us]
TransactionType                       str
Location                              str
DeviceID                              str
IP Address                            str
MerchantID                            str
Channel                               str
CustomerAge                         int64
CustomerOccupation                    str
TransactionDuration                 int64
LoginAttempts                       int64
AccountBalance                    float64
PreviousTransactionDate    datetime64[us]
dtype: object


In [11]:
# Compute per-account stats
latest_date = btx['TransactionDate'].max()
account_stats = btx.groupby('AccountID').agg(
    last_transaction=('TransactionDate','max'),
    avg_amount=('TransactionAmount','mean'),
    total_transactions=('TransactionID','count'),
    avg_balance=('AccountBalance','mean'),
    avg_login_attempts=('LoginAttempts','mean')
).reset_index()
account_stats['days_since_last'] = (latest_date - account_stats['last_transaction']).dt.days
account_stats['high_risk_account'] = account_stats['days_since_last'] > 60
print(f"Account stats shape: {account_stats.shape}")
print(f"High risk accounts: {account_stats['high_risk_account'].sum()}")


Account stats shape: (495, 8)
High risk accounts: 201


In [12]:
fig = px.histogram(btx, x='TransactionAmount', nbins=40, color_discrete_sequence=[PRIMARY],
                   template=TEMPLATE, title='Transaction Amount Distribution')
fig.show()
fig = px.histogram(btx, x='AccountBalance', nbins=40, color_discrete_sequence=[NEUTRAL],
                   template=TEMPLATE, title='Account Balance Distribution')
fig.show()


In [13]:
tt = btx['TransactionType'].value_counts().reset_index(); tt.columns=['type','count']
fig = px.pie(tt, values='count', names='type', title='Transaction Type', hole=0.3,
             color_discrete_sequence=[PRIMARY, WARNING], template=TEMPLATE)
fig.show()


In [14]:
ch = btx['Channel'].value_counts().reset_index(); ch.columns=['channel','count']
fig = px.bar(ch, x='channel', y='count', color_discrete_sequence=[NEUTRAL],
             template=TEMPLATE, title='Channel Distribution')
fig.show()


In [15]:
la = btx['LoginAttempts'].value_counts().sort_index().reset_index(); la.columns=['attempts','count']
fig = px.bar(la, x='attempts', y='count', color_discrete_sequence=[WARNING],
             template=TEMPLATE, title='Login Attempts Distribution')
fig.show()
fig = px.histogram(btx, x='TransactionDuration', nbins=30, color_discrete_sequence=[PRIMARY],
                   template=TEMPLATE, title='Transaction Duration Distribution')
fig.show()


In [16]:
occ = btx['CustomerOccupation'].value_counts().head(10).reset_index(); occ.columns=['occupation','count']
fig = px.bar(occ, x='occupation', y='count', color_discrete_sequence=[NEUTRAL],
             template=TEMPLATE, title='Top 10 Customer Occupations')
fig.show()
fig = px.histogram(btx, x='CustomerAge', nbins=20, color_discrete_sequence=[PRIMARY],
                   template=TEMPLATE, title='Customer Age Distribution')
fig.show()


In [17]:
fig = px.histogram(account_stats, x='days_since_last', nbins=30, color_discrete_sequence=[WARNING],
                   template=TEMPLATE, title='Days Since Last Transaction')
fig.show()
hr = account_stats['high_risk_account'].value_counts().reset_index(); hr.columns=['high_risk','count']
hr['high_risk'] = hr['high_risk'].map({True:'High Risk',False:'Normal'})
fig = px.bar(hr, x='high_risk', y='count', color='high_risk',
             color_discrete_map={'High Risk':RISK,'Normal':SAFE},
             template=TEMPLATE, title='High Risk Account Count')
fig.show()


In [18]:
# Save Part B
account_stats.to_csv('data/processed/bank_tx_activity.csv', index=False)
print(f"Saved bank_tx_activity.csv - shape: {account_stats.shape}")


Saved bank_tx_activity.csv - shape: (495, 8)
